In [ ]:
#rm -rf /kaggle/working/*

In [ ]:
#установка библиотек
!pip install -q ultralytics paddleocr matplotlib pillow

In [ ]:
#копирование датасета
!cp -r /kaggle/input/datasets/ottogalt/yolo-practica/yolo_export /kaggle/working/project/
!ls /kaggle/working/project

In [ ]:
#проверка графического процессора
import torch
from ultralytics import YOLO
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageOps
import numpy as np
import os
import json

print("CUDA доступна:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Текущий девайс:", torch.cuda.get_device_name(0))
else:
    print("Используется CPU")


In [ ]:
#обучение
yaml_path = '/kaggle/working/project/data.yaml'
model = YOLO('yolov8n.pt')  # можно заменить на yolov8s.pt или yolov8m.pt

model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=8,
    project='/kaggle/working',
    name='yolo_practica',
    val=True
)


In [ ]:
#вывод графиков модели
log_path = '/kaggle/working/yolo_practica/results.csv'
df = pd.read_csv(log_path)

metrics = {
    'train/box_loss': 'Box Loss (train)',
    'val/box_loss': 'Box Loss (val)',
    'train/cls_loss': 'Class Loss (train)',
    'val/cls_loss': 'Class Loss (val)',
    'train/dfl_loss': 'DFL Loss (train)',
    'val/dfl_loss': 'DFL Loss (val)',
    'metrics/precision(B)': 'Precision',
    'metrics/recall(B)': 'Recall',
    'metrics/mAP50(B)': 'mAP@0.5',
    'metrics/mAP50-95(B)': 'mAP@0.5:0.95'
}

for col, title in metrics.items():
    if col in df.columns:
        plt.figure(figsize=(10, 5))
        plt.plot(df['epoch'], df[col], label=title, color='blue')
        plt.xlabel('Epoch')
        plt.ylabel(title)
        plt.title(title + ' over Epoch')
        plt.legend()
        plt.grid(True)
        plt.show()

In [ ]:
#тестирование модели YOLOv8
import os
from ultralytics import YOLO
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import numpy as np

model = YOLO('/kaggle/working/yolo_practica/weights/best.pt')


test_folder = '/kaggle/working/project/images/test'
image_paths = [os.path.join(test_folder, f)
               for f in os.listdir(test_folder)
               if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f"Найдено тестовых изображений: {len(image_paths)}")


image_paths = image_paths[:30]


output_dir = '/kaggle/working/yolo_practica/test_results'
os.makedirs(output_dir, exist_ok=True)
print(f"Результаты будут сохранены в: {output_dir}\n")


for i, image_path in enumerate(image_paths):
    print(f"\n{'='*60}")
    print(f"Обработка {i+1}/{len(image_paths)}: {os.path.basename(image_path)}")
    
    try:
        with Image.open(image_path) as img_pil:
            img_pil = ImageOps.exif_transpose(img_pil)
            W, H = img_pil.size
            
            # YOLO инференс
            results = model.predict(source=np.array(img_pil), imgsz=640, conf=0.25, verbose=False)
            
            # Выводим изображение
            plt.figure(figsize=(10, 10))
            plt.imshow(img_pil)
            plt.axis('off')
            
            if len(results[0].boxes) > 0:
                print(f"Обнаружено объектов: {len(results[0].boxes)}")
                
                for j, box in enumerate(results[0].boxes):
                    cls_id = int(box.cls[0])
                    conf = float(box.conf[0])
                    label = model.names[cls_id]
                    
                    print(f"  {j+1}. {label.upper()} | Confidence: {conf:.2f}")
                    
                    # Координаты рамки
                    x_center, y_center, w_norm, h_norm = box.xywhn[0].tolist()
                    x1 = (x_center - w_norm/2) * W
                    y1 = (y_center - h_norm/2) * H
                    x2 = (x_center + w_norm/2) * W
                    y2 = (y_center + h_norm/2) * H
                    
                    # Рисуем рамку и подпись
                    plt.plot([x1, x2, x2, x1, x1], [y1, y1, y2, y2, y1], 'r-', linewidth=2)
                    plt.text(x1, y1-10, f'{label} {conf:.2f}', 
                             bbox=dict(facecolor='red', alpha=0.5),
                             fontsize=12, color='white')
            else:
                print("Объекты не обнаружены")
            
            
            output_path = os.path.join(output_dir, f"result_{os.path.basename(image_path)}")
            plt.tight_layout()
            plt.savefig(output_path, dpi=150, bbox_inches='tight', pad_inches=0.1)
            plt.show()
            plt.close()
            print(f"Сохранено: {output_path}")
            
    except Exception as e:
        print(f"Ошибка при обработке {image_path}: {str(e)}")
        continue

print(f"\n{'='*60}")
print(f"Обработано изображений: {len(image_paths)}")
print(f"Результаты сохранены в: {output_dir}")

In [ ]:
#тестирование модели OCR с YOLOv8 + сохранение результатов распознования текста в JSON
import os
import json
import numpy as np
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
from ultralytics import YOLO
import easyocr


model = YOLO('/kaggle/working/yolo_practica/weights/best.pt')


reader = easyocr.Reader(['en', 'ru'], gpu=True)  

test_folder = '/kaggle/working/project/images/test'
image_paths = [os.path.join(test_folder, f)
               for f in os.listdir(test_folder)
               if f.lower().endswith(('.jpg', '.jpeg', '.png'))]


image_paths = image_paths[:10]

print(f"Будет обработано первых {len(image_paths)} изображений")


output_dir = '/kaggle/working/yolo_practica/test_results_ocr'
output_json = '/kaggle/working/yolo_practica/test_results_ocr/json'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(output_json, exist_ok=True)
print(f"Результаты OCR будут сохранены в: {output_dir}\n")


all_results = []

for i, image_path in enumerate(image_paths):
    print(f"\n{'='*60}")
    print(f"Обработка {i+1}/{len(image_paths)}: {os.path.basename(image_path)}")
    
    image_result = {"filename": os.path.basename(image_path), "detections": []}

    try:
        with Image.open(image_path) as img_pil:
            img_pil = ImageOps.exif_transpose(img_pil)
            img_np = np.array(img_pil)
            W, H = img_pil.size

            
            results = model.predict(source=img_np, imgsz=640, conf=0.25, verbose=False)
            
            
            plt.figure(figsize=(10, 10))
            plt.imshow(img_pil)
            plt.axis('off')
            
            if len(results[0].boxes) > 0:
                print(f"Обнаружено объектов: {len(results[0].boxes)}")
                print("ДЕТЕКЦИИ с распознанным текстом:")

                for j, box in enumerate(results[0].boxes):
                    cls_id = int(box.cls[0])
                    conf = float(box.conf[0])
                    label = model.names[cls_id]

                    x_center, y_center, w_norm, h_norm = box.xywhn[0].tolist()
                    x1 = int((x_center - w_norm/2) * W)
                    y1 = int((y_center - h_norm/2) * H)
                    x2 = int((x_center + w_norm/2) * W)
                    y2 = int((y_center + h_norm/2) * H)

                    cropped = img_np[y1:y2, x1:x2]

                    ocr_result = reader.readtext(cropped, detail=0)
                    ocr_text = " | ".join(ocr_result) if ocr_result else "[не найдено]"

                    print(f"  {j+1}. {label.upper()} | Confidence: {conf:.2f}")
                    print(f"     Распознанный текст: {ocr_text}")

                    image_result["detections"].append({
                        "class": label,
                        "confidence": conf,
                        "text": ocr_text
                    })

                    plt.plot([x1, x2, x2, x1, x1], [y1, y1, y2, y2, y1], 'r-', linewidth=2)
                    plt.text(x1, y1-10, f'{label} {conf:.2f}', 
                             bbox=dict(facecolor='red', alpha=0.5),
                             fontsize=12, color='white')
            else:
                print("Объекты не обнаружены")

            output_path = os.path.join(output_dir, f"ocr_{os.path.basename(image_path)}")
            plt.tight_layout()
            plt.savefig(output_path, dpi=150, bbox_inches='tight', pad_inches=0.1)
            plt.show()
            plt.close()
            print(f"Сохранено: {output_path}")

            all_results.append(image_result)

    except Exception as e:
        print(f"Ошибка при обработке {image_path}: {str(e)}")
        continue

json_path = os.path.join(output_json, "ocr_results.json")
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, ensure_ascii=False, indent=4)

print(f"\n{'='*60}")
print(f"Обработано изображений: {len(image_paths)}")
print(f"Результаты сохранены в папке: {output_dir}")
print(f"JSON с результатами: {json_path}")

In [ ]:
#создание архива для сохранения результатов
!zip -r yolov8_practica.zip /kaggle/working/yolo_practica

In [ ]:
#скачивание zip-архива
from IPython.display import FileLink
FileLink(r'yolov8_practica.zip')